### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="home_credit_default_stability_1m",
    version_from_unique_name="home_credit_default_stability",
    version_comment="""
We randomly sub-sample the train data to 1 million rows. We follow TabReD and use random sub-sampling. The idea behind this instead of a time-based subsampling is to keep data from various time periods and model the distribution shift across the full time horizon.
""",
    # same as home_credit_default_stability.ipynb
    dataset_year="2024",
    domain_str="finance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/home-credit-credit-risk-model-stability",
    download_description="""
We get the data from the Kaggle competition.

kaggle competitions download -c home-credit-credit-risk-model-stability
mkdir -p local-data-warehouse/home_credit_default_stability && mv home-credit-credit-risk-model-stability.zip local-data-warehouse/home_credit_default_stability/ && cd local-data-warehouse/home_credit_default_stability/ && unzip home-credit-credit-risk-model-stability.zip && rm home-credit-credit-risk-model-stability.zip && rm -rf csv_files && rm -rf parquet_files/test && rm sample_submission.csv feature_definitions.csv
""",
    # References
    academic_reference_bibtex=r"""@misc{Herman2024HomeCreditCreditRiskModelStability,
  author = {Daniel Herman and Tomas Jelinek and Walter Reade and Maggie Demkin and Addison Howard},
  title  = {Home Credit - Credit Risk Model Stability},
  year   = {2024},
  howpublished = {\url{https://kaggle.com/competitions/home-credit-credit-risk-model-stability}},
  note   = {Kaggle competition}
}
""",
    academic_reference_bibtex_key="Herman2024HomeCreditCreditRiskModelStability",
    license="Kaggle Competition Rules",
    data_tags=["Non-IID", "Temporal"],
    curation_comments="""
We start with the data from Kaggle and follow the preprocessing from TabRed (https://github.com/yandex-research/tabred/tree/main/preprocessing#homecredit-default-stability-homecredit-20), which in turn follows two Kaggle solutions (https://www.kaggle.com/competitions/home-credit-credit-risk-model-stability/discussion/507946, https://www.kaggle.com/code/yuuniekiri/fork-of-home-credit-catboost-inference).

- We follow TabRed and use temporal splits for the task. The Kaggle experts used various strategies and most commonly StratifiedGroupKFold to align offline CV with the temporal-split data on the leaderboard. That is, the split used for training can be StratifiedGroupKFold, but not for testing.
- Note, the original competition was heavily influenced by metric hacking, which is not relevant for our offline benchmark tasks. Thus, results are not directly comparable ot transferable.
- We change the preprocessing from Kaggle and TabRed in specific steps: (A) we depart from the TabRed and Kaggle solutions in that we do not drop high-cardinality string columns, (B) we do not drop month and week, as it was only dropped due to the metric leak in the competition, (C) we only drop constant columns if they are constant w.r.t. NaN and non-NaN values, (D) we do not perform ordinal encoding and drop minor categories, (E) we do not sub-sample the data, (F) we keep the date column without transformations for the pipelines to handle, and (G) the code from TabRed is outdated and does not function the same anymore when it comes to down-casting string and categorical data of the data from Kaggle, we fix this by treating categorical data as categorical following the original Kaggle scripts, moreover, we had to remove getting the std() of a date column, which is not supported anymore.
- We drop case_id as it is already collapsed and thus does not contain extra information.
- We cast all object and string columns to be categorical. Note that they are anonymized and string preprocessing likely does not add a lot of value as a result.
- We drop name-related columns as they do not hold relevant information. These were otherwise dropped as high-cardinality string columns in TabRed or on Kaggle.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="target",
    problem_type="binary_classification",
    # There exists a custom metric see https://www.kaggle.com/competitions/home-credit-credit-risk-model-stability
    # However, we decide to not use it, as it was seen to be not stable and was able to be clearly exploited for modelling in the competition.
    objective_metric_name="roc_auc",
    stratify_on="target",
    time_on="date_decision",
)

## Preprocessing

Start by running `run_large_data_preprocessing.py` outside this notebook to get the merged data in place.

In [2]:
import pandas as pd

df = pd.read_parquet(dataset_mold.path / "merged_input_data.parquet")
print("Loaded data shape:", df.shape)

# Drop constant columns
df = df.drop(columns=[
    # ID column (already collapsed and thus no extra information
    "case_id",
    # Name columns that are missing or have no affect due to case-id related grouping and anonymization
    "last_name_4917606M",
    "first_name_4917606M",
    "last_name_4527232M",
    "first_name_4527232M",
    "last_employername_160M",
    "first_employername_160M",
])
df["date_decision"] = pd.to_datetime(df["date_decision"], format="%Y-%m-%d")
as_cat_type = [
    "target",
    *list(df.columns[
    (df.dtypes == "object") | (df.dtypes == "string")
    ])
]
df[as_cat_type] = df[as_cat_type].astype("category")
df = df.reset_index(drop=True)

Loaded data shape: (1526659, 719)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
    duplicate_column_check=False,
)


#### Dataset Overview
Rows: 1,526,659
Columns: 712
Use sampling: True (sample size: 152,666)
Get missing and unique counts per column...


missing/unique per-col:   0%|          | 0/712 [00:00<?, ?it/s]

Get example values per column...


examples per-col:   0%|          | 0/712 [00:00<?, ?it/s]

Get numeric feature statistics...


numeric stats:   0%|          | 0/598 [00:00<?, ?it/s]

Get cat stats...


cat stats:   0%|          | 0/114 [00:00<?, ?it/s]

Get target stats...
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['mean_totaloutstanddebtvalue_39A', 'max_debtoutstand_525A', 'max_totaloutstanddebtvalue_39A', 'min_totaloutstanddebtvalue_39A', 'std_annuity_853A', 'mean_outstandingamount_362A', 'max_outstandingamount_362A', 'std_credamount_590A', 'min_outstandingamount_362A', 'totalsettled_863A']
Rows remaining as candidates after top-10 filter: 212,416 (of 1,526,659)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,date_decision,MONTH,WEEK_NUM,target,actualdpdtolerance_344P,amtinstpaidbefduel24m_4187115A,annuity_780A,annuitynextmonth_57A,applicationcnt_361L,applications30d_658L,applicationscnt_1086L,applicationscnt_464L,applicationscnt_629L,applicationscnt_867L,avgdbddpdlast24m_3658932P,avgdbddpdlast3m_4187120P,avgdbdtollast24m_4525197P,avgdpdtolclosure24_3658938P,avginstallast24m_3658937A,avglnamtstart24m_4525187A,avgmaxdpdlast9m_3716943P,avgoutstandbalancel6m_4187114A,avgpmtlast12m_4525200A,cardtype_51L,clientscnt12m_3712952L,clientscnt3m_3712950L,clientscnt6m_3712949L,clientscnt_100L,clientscnt_1022L,clientscnt_1071L,clientscnt_1130L,clientscnt_157L,clientscnt_257L,clientscnt_304L,clientscnt_360L,clientscnt_493L,clientscnt_533L,clientscnt_887L,clientscnt_946L,cntincpaycont9m_3716944L,cntpmts24_3658933L,credamount_770A,credtype_322L,currdebt_22A,currdebtcredtyperange_828A,datefirstoffer_1144D,datelastinstal40dpd_247D,datelastunpaid_3546854D,daysoverduetolerancedd_3976961L,disbursedcredamount_1113A,disbursementtype_67L,downpmt_116A,dtlastpmtallstes_4499206D,eir_270L,equalitydataagreement_891L,firstclxcampaign_1125D,firstdatedue_489D,homephncnt_628L,inittransactionamount_650A,inittransactioncode_186L,isbidproduct_1095L,lastactivateddate_801D,lastapplicationdate_877D,lastapprcommoditycat_1041M,lastapprcommoditytypec_5251766M,lastapprcredamount_781A,lastapprdate_640D,lastcancelreason_561M,lastdelinqdate_224D,lastrejectcommoditycat_161M,lastrejectcommodtypec_5251769M,lastrejectcredamount_222A,lastrejectdate_50D,lastrejectreason_759M,lastrejectreasonclient_4145040M,lastst_736L,maininc_215A,maxannuity_159A,maxdbddpdlast1m_3658939P,maxdbddpdtollast12m_3658940P,maxdbddpdtollast6m_4187119P,maxdebt4_972A,maxdpdfrom6mto36m_3546853P,maxdpdinstldate_3546855D,maxdpdinstlnum_3546846P,maxdpdlast12m_727P,maxdpdlast24m_143P,maxdpdlast3m_392P,maxdpdlast6m_474P,maxdpdlast9m_1059P,maxdpdtolerance_374P,maxinstallast24m_3658928A,maxlnamtstart6m_4525199A,maxoutstandbalancel12m_4187113A,maxpmtlast3m_4525190A,mindbddpdlast24m_3658935P,mindbdtollast24m_4525191P,mobilephncnt_593L,monthsannuity_845L,numactivecreds_622L,numactivecredschannel_414L,numactiverelcontr_750L,numcontrs3months_479L,numincomingpmts_3546848L,numinstlallpaidearly3d_817L,numinstls_657L,numinstlsallpaid_934L,numinstlswithdpd10_728L,numinstlswithdpd5_4187116L,numinstlswithoutdpd_562L,numinstmatpaidtearly2d_4499204L,numinstpaid_4499208L,numinstpaidearly3d_3546850L,numinstpaidearly3dest_4493216L,numinstpaidearly5d_1087L,numinstpaidearly5dest_4493211L,numinstpaidearly5dobd_4499205L,numinstpaidearly_338L,numinstpaidearlyest_4493214L,numinstpaidlastcontr_4325080L,numinstpaidlate1d_3546852L,numinstregularpaid_973L,numinstregularpaidest_4493210L,numinsttopaygr_769L,numinsttopaygrest_4493213L,numinstunpaidmax_3546851L,numinstunpaidmaxest_4493212L,numnotactivated_1143L,numpmtchanneldd_318L,numrejects9m_859L,opencred_647L,pctinstlsallpaidearl3d_427L,pctinstlsallpaidlat10d_839L,pctinstlsallpaidlate1d_3546856L,pctinstlsallpaidlate4d_3546849L,pctinstlsallpaidlate6d_3546844L,pmtnum_254L,posfpd10lastmonth_333P,posfpd30lastmonth_3976960P,posfstqpd30lastmonth_3976962P,previouscontdistrict_112M,price_1097A,sellerplacecnt_915L,sellerplacescnt_216L,sumoutstandtotal_3546847A,sumoutstandtotalest_4493215A,totaldebt_9A,totalsettled_863A,totinstallast1m_4525188A,twobodfilling_608L,validfrom_1069D,assignmentdate_238D,assignmentdate_4527235D,birthdate_574D,contractssum_5085716L,dateofbirth_337D,days120_123L,days180_256L,days30_165L,days360_512L,days90_310L,description_5085714M,education_1103M,education_88M,firstquarter_103L,fourthquarter_440L,maritalst_385M,maritalst_893M,pmtaverage_3A,pmtaverage_4527227A,pmtcount_4527229L,pmtcount_693L,pmtscount_423L,pmtssum_45A,requesttype_4525192L,responsedate_1012D,responsedate_4527233D,responsedate_4917613D,secondquarter_766L,thirdquarter_1082L,max_actualdpd_943P,min_actualdpd_943P,mean_actualdpd_943P,std_actualdpd_943P,max_annuity_853A,min_annuity_853A,mean_annuity_853A,std_annuity_

In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,isbidproduct_1095L,bool,0.0,0.00,2.0,"False, True"
1,last_credacc_status_367L,category,1450057.0,94.98,6.0,"AC, CL, CA, PCL, PO, CR"
2,equalitydataagreement_891L,category,1448632.0,94.89,2.0,"True, False"
3,first_housetype_905L,category,1425841.0,93.40,6.0,"OWNED, PARENTAL, FLAT, COMPANY_FLAT, COOP_FLAT, STATE_FLAT"
4,last_relationshiptoclient_415T,category,1362974.0,89.28,10.0,"CHILD, SPOUSE, OTHER_RELATIVE, SIBLING, FRIEND, OTHER, PARENT, COLLEAGUE, NEIGHBOR, GRAND_PARENT"
5,cardtype_51L,category,1334968.0,87.44,2.0,"INSTANT, PERSONALIZED"
6,max_isdebitcard_527L,category,1182972.0,77.49,2.0,"False, True"
7,min_isdebitcard_527L,category,1182972.0,77.49,2.0,"False, True"
8,first_familystate_726L,category,1026869.0,67.26,5.0,"MARRIED, SINGLE, WIDOWED, DIVORCED, LIVING_WITH_PARTNER"
9,first_empl_industry_691L,category,1004423.0,65.79,24.0,"OTHER, GOVERNMENT, EDUCATION, TRADE, HEALTH, MANUFACTURING, AGRICULTURE, TRANSPORTATION, MINING, CATERING"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
MONTH,152666.0,2.019363e+05,4.473666e+01,2.019010e+05,2.020100e+05
WEEK_NUM,152666.0,4.066191e+01,2.380451e+01,0.000000e+00,9.100000e+01
actualdpdtolerance_344P,111005.0,3.872798e-02,5.548453e+00,0.000000e+00,1.816000e+03
amtinstpaidbefduel24m_4187115A,96329.0,5.584494e+04,7.159737e+04,0.000000e+00,1.007363e+06
annuity_780A,152666.0,4.027584e+03,2.998589e+03,1.494000e+02,5.197300e+04
annuitynextmonth_57A,152666.0,1.440332e+03,2.819701e+03,0.000000e+00,7.092420e+04
applicationcnt_361L,152666.0,1.310049e-05,3.619449e-03,0.000000e+00,1.000000e+00
applications30d_658L,152666.0,1.397561e-01,4.885494e-01,0.000000e+00,1.900000e+01
applicationscnt_1086L,152666.0,4.187638e-01,2.732703e+00,0.000000e+00,2.580000e+02
applicationscnt_464L,152666.0,1.098660e+00,9.869553e+00,0.000000e+00,2.370000e+02


In [7]:
# Categorical Feature Statistics
cat_stats

value   count  \
column                                rank                                    
cardtype_51L                          1                        <NA>  133419   
                                      2                     INSTANT   19016   
                                      3                PERSONALIZED     231   
credtype_322L                         1                         COL   98735   
                                      2                         CAL   34612   
                                      3                         REL   19318   
                                      4                        <NA>       1   
date_decision                         1         2019-11-29 00:00:00     838   
                                      2         2019-11-30 00:00:00     807   
                                      3         2019-12-29 00:00:00     690   
                                      4         2019-12-28 00:00:00     679   
                                      5         2019-11-16 00:00:00     618   
description_5085714M                  1                    a55475b1  131729   
                                      2                    2fc785b2   18320   
                                      3                        <NA>    2617   
disbursementtype_67L                  1                         SBA  113868   
                                      2                         GBA   34613   
                                      3                          DD    4089   
                                      4                        <NA>      96   
education_1103M                       1                    a55475b1   85978   
                                      2                    6b2ae0fa   45225   
                                      3                    717ddd49   13443   
                                      4                    39a0853f    4805   
                                      5                        <NA>    2617   
education_88M                         1                    a55475b1  148471   
                                      2                        <NA>    2617   
                                      3                    6b2ae0fa    1185   
                                      4                    717ddd49     334   
                                      5                    a34a13c8      46   
equalitydataagreement_891L            1                        <NA>  144743   
                                      2                        True    7379   
                                      3                       False     544   
first_cancelreason_3545846M           1                    a55475b1   76662   
                                      2                        <NA>   30363   
                                      3                 P94_109_143   30053   
                                      4                   P30_86_84    2170   
                                      5                 P85_114_140    2136   
first_classificationofcontr_13M       1                    ea6782cc  115839   
                                      2                        <NA>   14102   
                                      3                    a55475b1   12221   
                                      4                    01f63ac8    6322   
                                      5                    00135d9c    2301   
first_classificationofcontr_400M      1                    ea6782cc   51991   
                                      2                    a55475b1   37503   
                                      3                        <NA>   14102   
                                      4                    01f63ac8    6874   
                                      5                    42a42e75    5779   
first_collater_typofvalofguarant_298M 1                    9a0c095e   98392   
                                      2                    8fd95e4b   27848   
                                 

In [8]:
# Target Distribution
target_df

,count,pct
target,,
0,1478665,96.86
1,47994,3.14


## Task Curation

In [9]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import subsample_temporal

split_time = pd.Timestamp("2020-05-01")
# Test: all data from 2020-05-01 onwards
test_idx = df.index[
    df["date_decision"] >= split_time
].to_numpy().tolist()

# Train: all data before that
train_idx = df.index[
    df["date_decision"] < split_time
].to_numpy().tolist()

df, train_idx, test_idx = subsample_temporal(
    df=df,
    train_idx=train_idx,
    test_idx=test_idx,
    stratify_on=task_mold.stratify_on,
)

# Size and class distribution checks
print("Train size:", len(train_idx), " | Test size:", len(test_idx))
print("Train target distribution:\n", df.loc[train_idx, task_mold.target_column_name].value_counts(normalize=True))
print("Test target distribution:\n", df.loc[test_idx, task_mold.target_column_name].value_counts(normalize=True))

splits = {
    0: {
        0: (train_idx, test_idx),
    }
}

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We follow TabRed and simulate a use case of a model being refit every 4 to 5 months. We create one test split using 4,5 months of data (2020-05-01 to 2020-10-05) for testing and all the previous data for training.",
    splits=splits,
    time_horizon=5, # round up
    time_horizon_unit="months",
)

Train size: 1000000  | Test size: 224927
Train target distribution:
 target
0    0.966924
1    0.033076
Name: proportion, dtype: float64
Test target distribution:
 target
0    0.978046
1    0.021954
Name: proportion, dtype: float64


## Export

In [10]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to home_credit_default_stability/versions/019d308f-f814-7a3e-bf8f-508273a0e8f1
019d308f-f814-7a3e-bf8f-508273a0e8f1
63494a10e58d15d1abea2c64aca6e107f942773f687782051b10185f990ca754
